# **Fairness-Aware Age Estimation: Bias Detection and Mitigation - Part I**
UB Master in Fundamental Principels of Data Science (2025-2026)

Author: Julio C. S. Jacques Junior

Last modified: Jan, 2026.

---

# **Practical Sessions Objectives**

- Explore different deep learning concepts incrementally.

- Define new strategies to **improve accuracy** (overall and per attribute) **while minimizing different bias scores**.

- Based on your experiments, **be able to provide a strong analysis and discussion of the results** when delivering your report.

- As you will see, the problem is basically solved in this starting kit. However, **we expect you to go beyond the starting kit**.

- In summary, **you are expected to** experiment with some basic concepts such as:
  - hyperparameter tuning (e.g., learning rate, batch size, optimizer);
  - modifying the model or the regression head of the provided architecture, using new regularizers (e.g., Dropout layers) or fully connected layers of different dimensions (instead of using the model exactly as defined in the starting kit);
  - training strategies (e.g., using a different training/validation split or multi-stage training strategies; freezing/unfreezing different layers of the model during training, etc.);
  - training from scratch instead of using transfer learning and compare both strategies, for example;
  - using different pre-trained models (e.g., some model pretrained on Faces dataset instead of ImageNet);
  - using a different backbone (e.g., ResNet instead of ViT).
  - propose some multimodal architecture (e.g., combining image and metadata information) to improve model accuracy and mitigate the bias problem.
  - play with **data augmentation (part II)** and **custom loss (part III)** to address the bia problem.

- In the end, the goal of this exercise is to become familiar with some basic concepts related to Computer Vision / Deep Learning and the scientific method used in research. To do so, you are expected to evaluate different models and cases in a scientific manner before making a final decision about which model you would recommend as the best one, based on your experiments. As a simple example, you can compare **“Model A” vs. “Model B”** with respect to accuracy, bias metrics, training time, etc., where “A” and “B” are defined based on some of the items mentioned above.

- **Do not make only minor changes to the provided code** as your final solution. You will not be evaluated based on the accuracy your model achieves, but rather on your creativity and on how you design the experiments, report, and discuss the results (using the report document template detailed on Virtual Campus).

- You are expected to train different models and evaluate different strategies using the training and **validation** sets.

- **When you are satisfied with your defined model, hyperparemeters and results on the validation set, you can obtain the final results on the test set**. We consider this a good exercise that simulates a real-world scenario where test labels are not accessible (this is also useful and good practice to avoid overfitting your model on the test data).

> **If you have any questions or find any bugs, do not hesitate to contact me.** I hope you enjoy working on this task, which was carefully designed for this course.




## Preliminary Instructions

- Check Colab GPU usage instructions [here](https://research.google.com/colaboratory/faq.html#gpu-availability).

- Some parts of the code download data or models from our server. Occasionally, you may encounter a "file not found" error due to temporary server instability. If this happens, please try again. If the error persists, contact me for assistance.


## Checking the pytorch version
 - This notebook was successfully tested on version = 2.9.0+cu126

In [ ]:
import torch
print(torch.__version__)

In [ ]:
# installing torchinfo to print model summary
!pip install torchinfo

## Importing required libraries

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import csv
from PIL import Image
from timm import create_model
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
import copy
from tqdm import tqdm

# **Downloading the Appa-Real Dataset**

- In these practical sessions, we will use the [Appa-Real Dataset](http://chalearnlap.cvc.uab.es/challenge/13/track/13/description/), which consists of face images, age labels, and accompanying metadata. The dataset provides both real and perceived age labels, but in these sessions, we will focus only on the perceived age labels.

- Original RGB images (cropped faces) have pixel values in the range [0, 255], and labels represent ages in the approximate range of 0.9 to 90 years. Later, you will see that **we re-scale the age values to the range [0, 1]** to simplify operations (e.g., training), and eventually re-scale the output back during evaluation **using a predefined normalization factor**.

- Metadata is also provided with the dataset:
  - **Gender:** male / female  
  - **Ethnicity:** asian / afroamerican / caucasian  
  - **Facial expression:** neutral / slightly happy / happy / other

- You can see the data we have downloaded and the structure of Colab by clicking on 'Files', on the left side (<--) of this interface.  

- The data is divided into **train**, **validation**, and **test** sets. However, **you MUST NOT use the Test set to tune hyperparameters, training strategies, or to improve/define your model(s). All preliminary experiments MUST be conducted using the TRAINING and VALIDATION data.** Test data will be used only to generate the final results after the best model is defined based on the validation set.

  - For example, imagine you evaluate models "A", "B", "C", and "D" on the validation data. If the best results are obtained using model "C", then **model "C" is your final model**, and only then you compute and report the final results on the Test set.

- You can train your models using the provided train and validation sets, as done in this starting kit, or define your own train/validation split. For instance, you could merge the provided train and validation data and randomly split them differently. You are free to design any training strategy. **Your creativity will have a significant impact on evaluation.**


In [ ]:
from zipfile import ZipFile

# downloading the data
!wget https://data.chalearnlap.cvc.uab.cat/Colab_MFPDS/2025/appa-real-dataset_v2.zip

with ZipFile('appa-real-dataset_v2.zip','r') as zip:
   zip.extractall()
   print('Data decompressed successfully')

# removing the .zip file after extraction to clean space
!rm appa-real-dataset_v2.zip

# **Mount Google Drive to save the trained model on the cloud**

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')
# Note, the default path will be: '/content/gdrive/MyDrive/'
# In my case, the final path will be: '/content/gdrive/MyDrive/temp/' as I
# created a '/temp/' folder in my google drive for this purpose.

# **Defining the Data Loader Class**
- In this example, metadata information is loaded but not used. Future implementations can take benefit of it.
- Note that age labels are divided by 100 (assuming 100 is the max age found in the dataset) so that the age values can be normalized to be in the range of 0 and 1. This way, we can add a sigmoid activation in the last layer of our model.

In [ ]:
class AgeEstimationDataset(Dataset):
    def __init__(self, image_dir, csv_file, transform=None):
        self.image_dir = image_dir
        self.data_info = pd.read_csv(csv_file)
        self.base_transforms = base_transforms
        self.age_normalization_factor = 100; # used to normalize age labels

    def __len__(self):
        return len(self.data_info)

    def __normalization_factor__(self):
        return self.age_normalization_factor

    def __getitem__(self, idx):
        image_id = f"{self.data_info.iloc[idx, 0]:06d}.jpg"  # Format image ID
        image_path = os.path.join(self.image_dir, image_id)
        image = Image.open(image_path).convert("RGB")  # Load image as RGB

        raw_age = float(self.data_info.iloc[idx, 1])
        # normalizing age labes (by 100) to be between 0 and 1 (assuming 100 is the max age)
        age = raw_age / self.age_normalization_factor
        metadata = self.data_info.iloc[idx, 2:].tolist()  # Extract metadata as list

        image = self.base_transforms(image)

        return image, torch.tensor(age, dtype=torch.float32), metadata

# **Defining the base image transformations**

In [ ]:
base_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])

# **Loading the Train and Validation sets**

In [ ]:
# Create dataset and dataloader (train set):
dataset_train = AgeEstimationDataset("train_data", "labels_metadata_train.csv", transform=base_transforms)
dataloader_train = DataLoader(dataset_train, batch_size=32, shuffle=True, num_workers=2)
print(f"Total number of train samples: {len(dataloader_train.dataset)}")

# Create dataset and dataloader (validation set):
dataset_valid = AgeEstimationDataset("valid_data", "labels_metadata_valid.csv", transform=base_transforms)
dataloader_valid = DataLoader(dataset_valid, batch_size=32, shuffle=True, num_workers=2)
print(f"Total number of valid samples: {len(dataloader_valid.dataset)}")

# **Visualizing some train samples**

In [ ]:
import random

# Function to display image samples
def display_samples(dataset, num_samples=3):
    fig, axes = plt.subplots(1, num_samples, figsize=(11, 5))
    for i in range(num_samples):
        random_index = random.randint(0, len(dataset) - 1)
        image, age, metadata = dataset[random_index]

        image = image * 0.5 + 0.5  # Denormalize
        image = image.permute(1, 2, 0).numpy() if isinstance(image, torch.Tensor) else np.array(image)

        axes[i].imshow(image)
        axes[i].axis("off")
        # denormalizing the age values to plot the original labels
        axes[i].set_title(f"Age: {age*dataset.__normalization_factor__():.2f}\n{', '.join(metadata)}")
    plt.show()

display_samples(dataset_train, num_samples=3)

# **Plotting age, gender, ethnicity, and emotion distributions in the training set**

In [ ]:

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load dataset
df = pd.read_csv("labels_metadata_train.csv")
sns.set(style="whitegrid")

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Age distribution
axes[0, 0].hist(df['age'], bins=20, color='skyblue', edgecolor='black')
axes[0, 0].set_title("Age Distribution")
axes[0, 0].set_xlabel("Age")
axes[0, 0].set_ylabel("Count")

# 2. Gender distribution
gender_df = df['gender'].value_counts().reset_index()
gender_df.columns = ['gender', 'count']
# Add a dummy hue column for compatibility
gender_df['hue'] = gender_df['gender']
sns.barplot(data=gender_df, x='gender', y='count', hue='hue', palette='pastel', dodge=False, ax=axes[0, 1], legend=False)
axes[0, 1].set_title("Gender Distribution")
axes[0, 1].set_ylabel("Count")
axes[0, 1].set_xlabel("Gender")
for i, v in enumerate(gender_df['count']):
    axes[0, 1].text(i, v + 0.5, str(v), ha='center')

# 3. Ethnicity distribution
ethnicity_df = df['ethnicity'].value_counts().reset_index()
ethnicity_df.columns = ['ethnicity', 'count']
ethnicity_df['hue'] = ethnicity_df['ethnicity']
sns.barplot(data=ethnicity_df, x='ethnicity', y='count', hue='hue', palette='pastel', dodge=False, ax=axes[1, 0], legend=False)
axes[1, 0].set_title("Ethnicity Distribution")
axes[1, 0].set_ylabel("Count")
axes[1, 0].set_xlabel("Ethnicity")
axes[1, 0].tick_params(axis='x', rotation=45)
for i, v in enumerate(ethnicity_df['count']):
    axes[1, 0].text(i, v + 0.5, str(v), ha='center')

# 4. Emotion distribution
emotion_df = df['emotion'].value_counts().reset_index()
emotion_df.columns = ['emotion', 'count']
emotion_df['hue'] = emotion_df['emotion']
sns.barplot(data=emotion_df, x='emotion', y='count', hue='hue', palette='pastel', dodge=False, ax=axes[1, 1], legend=False)
axes[1, 1].set_title("Emotion Distribution")
axes[1, 1].set_ylabel("Count")
axes[1, 1].set_xlabel("Emotion")
axes[1, 1].tick_params(axis='x', rotation=45)
for i, v in enumerate(emotion_df['count']):
    axes[1, 1].text(i, v + 0.5, str(v), ha='center')

plt.tight_layout()
plt.show()


# **Loading the pretrained ViT baselone model and adapting it to our problem**

- ViT normally outputs class scores for classification tasks; here, we adapt it for regression by setting **num_classes=1**.

> **Note:** This notebook is intended as a starting point. For your deliverables, avoid making only minor modifications. Instead, explore your creativity and try more substantial improvements.



In [ ]:
# Vision Transformer Model for Age Prediction (pretrained on ImageNet)
# https://pytorch.org/vision/main/models/vision_transformer.html
# https://huggingface.co/docs/transformers/main/en//model_doc/vit
class AgeEstimationViT(nn.Module):
    def __init__(self):
        super(AgeEstimationViT, self).__init__()
        self.vit = create_model("vit_base_patch16_224", pretrained=True, num_classes=1) # num_classes=1 as we want to regress a single (age value)
        self.activation = nn.Sigmoid()  # Added Sigmoid activation

    def forward(self, x):
        x = self.vit(x)
        return self.activation(x)  # Apply Sigmoid activation

In [ ]:
# creating the model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AgeEstimationViT().to(device)

In [ ]:
# print model summary
from torchinfo import summary
summary(model, input_size=(1, 3, 224, 224))  # Adjust based on your model

# Defining an auxiliary function to plot the training history

In [ ]:
# Function to plot training curves
def plot_training_curves(train_losses, val_losses):
    plt.figure(figsize=(8, 5))
    plt.plot(train_losses, label='Train Loss')
    plt.plot(val_losses, label='Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Training & Validation Loss')
    plt.legend()
    plt.grid()
    plt.show()

# **Defining the Training function**
- Our training code includes **early stopping** and **automatic saving of the best model**. Early stopping monitors the validation loss during training and halts training if the model stops improving for a specified number of epochs, helping to prevent overfitting and save computational resources. At the same time, the model with the lowest validation loss is automatically saved, ensuring that we keep the best-performing version for evaluation or deployment.

In [ ]:
# Training function with early stopping and model saving
def train_model(model, dataloaders, criterion, optimizer, scheduler, num_epochs, patience, model_path):
    best_model_wts = copy.deepcopy(model.state_dict())
    best_loss = float("inf")
    early_stopping_counter = 0
    train_losses = []
    val_losses = []

    for epoch in range(num_epochs):
        print(f'Epoch {epoch+1}/{num_epochs}')

        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
            else:
                model.eval()

            running_loss = 0.0

            with tqdm(dataloaders[phase], desc=f"{phase.capitalize()} Epoch {epoch+1}") as t:
                for inputs, labels, _ in t:
                    inputs, labels = inputs.to(device), labels.to(device)

                    optimizer.zero_grad()

                    with torch.set_grad_enabled(phase == 'train'):
                        outputs = model(inputs)
                        loss = criterion(outputs.view_as(labels), labels)

                        if phase == 'train':
                            loss.backward()
                            optimizer.step()

                    running_loss += loss.item() * inputs.size(0)
                    t.set_postfix(loss=loss.item())

            epoch_loss = running_loss / len(dataloaders[phase].dataset)
            print(f'{phase} Loss: {epoch_loss:.6f}')

            if phase == 'train':
                train_losses.append(epoch_loss)
            else:
                val_losses.append(epoch_loss)
                scheduler.step(epoch_loss)

                if epoch_loss < best_loss:
                    best_loss = epoch_loss
                    best_model_wts = copy.deepcopy(model.state_dict())
                    early_stopping_counter = 0
                    # Save best model during training
                    print("saving best model...")
                    torch.save(best_model_wts, model_path)
                else:
                    early_stopping_counter += 1
                    if early_stopping_counter >= patience:
                        print("Early stopping triggered.")
                        model.load_state_dict(best_model_wts)
                        plot_training_curves(train_losses, val_losses)
                        return model

    model.load_state_dict(best_model_wts)
    plot_training_curves(train_losses, val_losses)
    return model

# **Training the Model (or Loading a Pre-Trained Model)**

- If `MODEL_TRAIN = False`, the code will load a pre-trained model that was trained using the provided training code.

- To perform training (fine tunning the model pretrained on ImageNet), set `MODEL_TRAIN = True`.  

  - The code uses **early stopping** with `patience = 10`, meaning that training will stop if the validation loss does not improve for 10 consecutive epochs.  
  - The **loss function** is **Mean Squared Error (MSE)** (`criterion = nn.MSELoss()`).  
  - Training hyperparameters include: **learning rate = 1e-5**, **batch size = 32**, and **number of epochs = 50** (adjustable depending on your Colab runtime).  
  - The **model checkpoint callback** saves the best model based on validation loss. Best model will be saved on Google Drive, given the path defined by `model_filename`.
  - Other hyperparameters you can experiment with include the optimizer, loss function, learning rate, batch size, and number of epochs.  



In [ ]:
#==================
MODEL_TRAIN = True
#==================

model_filename = "/content/gdrive/MyDrive/temp/best_age_estimation_model.pth"

# model hyperparameters
num_epochs = 50
patience = 10
criterion = nn.MSELoss()
optimizer = Adam(model.parameters(), lr=1e-5)
scheduler = ReduceLROnPlateau(optimizer, mode='min', patience=patience)

# data loaders
dataloaders = {"train": dataloader_train, "val": dataloader_valid}  # Assuming split dataset

if (MODEL_TRAIN):
  best_model = train_model(model, dataloaders, criterion, optimizer, scheduler, num_epochs, patience, model_filename)
else:
  # download the pretrained model and training curves
  !wget https://data.chalearnlap.cvc.uab.cat/Colab_MFPDS/2026/best_age_estimation_model.pth
  !wget https://data.chalearnlap.cvc.uab.cat/Colab_MFPDS/2026/ViTbaseline_train_history.png

  # display the training curves
  from PIL import Image
  import matplotlib.pyplot as plt
  img = Image.open('ViTbaseline_train_history.png')  # replace with your file path
  plt.imshow(img)
  plt.axis('off')


# **Defining an auxiliary function to evaluate the model**
- We evaluate the model using the Mean Absolute Error as matric (MAE)

- The train/validation labels were re-scaled to be in the range of [0,1] for training and the predictions will be in the same range [0,1] (given the Sigmoid Activation in the last layer). Thus, we re-scale them back (to be in the range of "ages") to facilitate the analysis.

- The function also generates a ***submission file*** (.zip) that can be uploaded to our challenge, when the test set is being evaluated.

- The generated submission file can be downloaded from the left side (<--) of this interface.

In [ ]:
# Function to make predictions on test set and compute MSE
def predict_and_evaluate(model_path, test_dataset, output_zip=None, batch_size=32, output_csv="predictions.csv"):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = AgeEstimationViT().to(device)
    model.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))
    model.eval()

    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
    predictions = []
    actual_ages = []

    with torch.no_grad():
        for images, labels, metadata in tqdm(test_loader, desc="Predicting"):
            images = images.to(device)
            outputs = model(images).squeeze().cpu().numpy()
            labels = labels.cpu().numpy()

            predictions.extend(outputs * test_dataset.__normalization_factor__())
            actual_ages.extend(labels * test_dataset.__normalization_factor__())

    mae = np.mean(np.abs(np.array(predictions) - np.array(actual_ages)))
    if output_zip is not None:
      print(f"\n=======\nMean Absolute Error on Test Set: {mae:.4f}")
    else:
      print(f"\n=======\nMean Absolute Error on Validation Set: {mae:.4f}")


    # Only create ZIP if output_zip is provided
    if output_zip is not None:
        # Save predictions to CSV without headers
        with open(output_csv, mode='w', newline='') as file:
            writer = csv.writer(file)
            for pred in predictions:
                writer.writerow([pred])

        with ZipFile(output_zip, 'w') as zipf:
            zipf.write(output_csv, os.path.basename(output_csv))
        print(f"Predictions saved to {output_csv} and compressed as {output_zip}")

    return predictions, mae

# **Loading the Saved Model and Making Predictions on the Validation Set**



In [ ]:
if(MODEL_TRAIN == False):
  # load the baseline (pre-trained) model without data augmentation
  model_filename = "best_age_estimation_model.pth"

# Run prediction and compute MAE
predictions, mae = predict_and_evaluate(model_filename, dataset_valid,output_zip=None, batch_size=32,output_csv=None)



---



# **Generating the submission fie (on the test set) for our challenge**

- Loading the Saved Model and Making Predictions on the Test Set

- The following cells are generating predictions (and evaluating them) on the **Test set** so that we can create our submission file to be uploaded to our age estimation challenge.

- **Do not evaluate your model on the Test set when defining your model, training strategy, or hyperparameters.** For this, use the Validation set.

In [ ]:
# Create dataset and dataloader (test set):
dataset_test = AgeEstimationDataset("test_data", "labels_metadata_test.csv", transform=base_transforms)
print(f"Total number of test samples: {len(dataset_test)}")


In [ ]:
if(MODEL_TRAIN == False):
  # load the baseline (pre-trained) model without data augmentation
  model_filename = "best_age_estimation_model.pth"

# Run prediction and compute MAE
predictions, mae = predict_and_evaluate(model_filename, dataset_test, "predictionsViTbaseline.zip")

